# 📘 Notebook 1 — Programmation Orientée Objet Python
### Préparation Entretien Senior AI Engineer — BNP Paribas

---
## Sommaire
1. [Fondamentaux POO](#fondamentaux)
2. [Interfaces & Classes Abstraites](#interfaces)
3. [Design Pattern — Factory](#factory)
4. [Design Pattern — Strategy](#strategy)
5. [Design Pattern — Adapter](#adapter)
6. [Design Pattern — Decorator](#decorator)
7. [Design Pattern — Singleton](#singleton)
8. [Design Pattern — Observer](#observer)
9. [SOLID Principles](#solid)
10. [Dataclasses & Protocols](#dataclasses)
11. [Context Managers & Generators](#context)
12. [Metaclasses](#meta)
13. [Questions d'entretien fréquentes](#questions)


---
## 1. Fondamentaux POO <a name='fondamentaux'></a>

### Les 4 piliers : Encapsulation, Héritage, Polymorphisme, Abstraction

In [ ]:
# ============================================================
# ENCAPSULATION — Contrôler l'accès aux attributs
# ============================================================
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner          # public
        self._balance = balance     # protected (convention)
        self.__pin = "1234"         # private (name mangling -> _BankAccount__pin)

    @property
    def balance(self) -> float:
        """Getter — accès contrôlé"""
        return self._balance

    @balance.setter
    def balance(self, amount: float):
        """Setter — validation incluse"""
        if amount < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = amount

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        print(f"Deposited {amount}. New balance: {self._balance}")

    def __repr__(self):
        return f"BankAccount(owner={self.owner!r}, balance={self._balance})"

acc = BankAccount("Alice", 1000)
acc.deposit(500)
print(acc)
print(f"Balance via property: {acc.balance}")

# Name mangling
print(f"Private attribute accessible via: {acc._BankAccount__pin}")


In [ ]:
# ============================================================
# HÉRITAGE & POLYMORPHISME
# ============================================================
from abc import ABC, abstractmethod
from typing import List

class Shape(ABC):
    """Classe abstraite de base"""
    color: str = "white"

    @abstractmethod
    def area(self) -> float:
        """Chaque forme DOIT implémenter area()"""
        ...

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:
        # Polymorphisme : comportement différent selon la sous-classe
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        import math
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        import math
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

# Polymorphisme en action
shapes: List[Shape] = [Circle(5), Rectangle(4, 6), Circle(3)]
for shape in shapes:
    print(shape.describe())  # Même interface, comportements différents


---
## 2. Interfaces & Classes Abstraites <a name='interfaces'></a>

> En Python, les interfaces sont simulées via `ABC` (Abstract Base Classes) ou `Protocol` (duck typing structurel).

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

# ============================================================
# ABC — Interface formelle (héritage nominal)
# ============================================================
class DataLoader(ABC):
    """Interface pour charger des données ML"""

    @abstractmethod
    def load(self, path: str) -> dict:
        """Charge les données depuis path"""
        ...

    @abstractmethod
    def validate(self, data: dict) -> bool:
        """Valide le schéma des données"""
        ...

    def load_and_validate(self, path: str) -> dict:
        """Template method — logique commune"""
        data = self.load(path)
        if not self.validate(data):
            raise ValueError(f"Invalid data from {path}")
        return data

class CSVLoader(DataLoader):
    def load(self, path: str) -> dict:
        print(f"Loading CSV from {path}")
        return {"data": [1, 2, 3], "format": "csv"}

    def validate(self, data: dict) -> bool:
        return "data" in data

loader = CSVLoader()
result = loader.load_and_validate("data.csv")
print(result)

# ============================================================
# Protocol — Interface structurelle (duck typing)
# ============================================================
@runtime_checkable
class Serializable(Protocol):
    def serialize(self) -> str: ...
    def deserialize(self, data: str) -> None: ...

class ModelConfig:
    def __init__(self, lr: float, epochs: int):
        self.lr = lr
        self.epochs = epochs

    def serialize(self) -> str:
        import json
        return json.dumps({"lr": self.lr, "epochs": self.epochs})

    def deserialize(self, data: str) -> None:
        import json
        d = json.loads(data)
        self.lr = d["lr"]
        self.epochs = d["epochs"]

config = ModelConfig(0.001, 10)
print(isinstance(config, Serializable))  # True — sans héritage !
print(config.serialize())


---
## 3. Design Pattern — Factory <a name='factory'></a>

> **But** : Déléguer la création d'objets à une classe/méthode dédiée. Découple le code client des classes concrètes.

**Use case ML** : Instancier dynamiquement différents modèles selon un paramètre de config.

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, Type, Any

# ============================================================
# FACTORY METHOD PATTERN
# ============================================================
class BaseModel(ABC):
    """Interface commune pour tous les modèles ML"""
    def __init__(self, **kwargs):
        self.params = kwargs

    @abstractmethod
    def fit(self, X, y): ...

    @abstractmethod
    def predict(self, X): ...

    def __repr__(self):
        return f"{type(self).__name__}({self.params})"

class LogisticRegressionModel(BaseModel):
    def fit(self, X, y):
        print(f"Training LogisticRegression with C={self.params.get('C', 1.0)}")
    def predict(self, X):
        return [0] * len(X)

class RandomForestModel(BaseModel):
    def fit(self, X, y):
        print(f"Training RandomForest with n_estimators={self.params.get('n_estimators', 100)}")
    def predict(self, X):
        return [1] * len(X)

class XGBoostModel(BaseModel):
    def fit(self, X, y):
        print(f"Training XGBoost with lr={self.params.get('learning_rate', 0.1)}")
    def predict(self, X):
        return [0] * len(X)

# ============================================================
# FACTORY — Registre dynamique (pattern Registry)
# ============================================================
class ModelFactory:
    _registry: Dict[str, Type[BaseModel]] = {
        "logistic": LogisticRegressionModel,
        "random_forest": RandomForestModel,
        "xgboost": XGBoostModel,
    }

    @classmethod
    def register(cls, name: str, model_class: Type[BaseModel]) -> None:
        """Permet d'enregistrer de nouveaux modèles à la volée"""
        cls._registry[name] = model_class

    @classmethod
    def create(cls, model_type: str, **kwargs) -> BaseModel:
        if model_type not in cls._registry:
            raise ValueError(f"Unknown model: {model_type}. Available: {list(cls._registry)}")
        return cls._registry[model_type](**kwargs)

    @classmethod
    def available_models(cls):
        return list(cls._registry.keys())

# Usage
config = {"model_type": "random_forest", "n_estimators": 200}
model = ModelFactory.create(**config)
print(model)
model.fit([[1, 2], [3, 4]], [0, 1])

# Enregistrer un nouveau modèle dynamiquement
class SVMModel(BaseModel):
    def fit(self, X, y): print("Training SVM")
    def predict(self, X): return [0] * len(X)

ModelFactory.register("svm", SVMModel)
print(f"Available models: {ModelFactory.available_models()}")


---
## 4. Design Pattern — Strategy <a name='strategy'></a>

> **But** : Définir une famille d'algorithmes interchangeables. Permet de changer l'algorithme utilisé à runtime.

**Use case ML** : Changer la stratégie de prétraitement, la fonction de loss, ou la méthode d'évaluation.

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Callable
import math

# ============================================================
# STRATEGY PATTERN — Stratégies de normalisation
# ============================================================
class NormalizationStrategy(ABC):
    @abstractmethod
    def normalize(self, data: List[float]) -> List[float]: ...

    @abstractmethod
    def name(self) -> str: ...

class MinMaxNormalization(NormalizationStrategy):
    def normalize(self, data: List[float]) -> List[float]:
        mn, mx = min(data), max(data)
        if mx == mn:
            return [0.0] * len(data)
        return [(x - mn) / (mx - mn) for x in data]

    def name(self) -> str:
        return "MinMax [0,1]"

class ZScoreNormalization(NormalizationStrategy):
    def normalize(self, data: List[float]) -> List[float]:
        n = len(data)
        mean = sum(data) / n
        std = math.sqrt(sum((x - mean) ** 2 for x in data) / n)
        return [(x - mean) / std if std > 0 else 0.0 for x in data]

    def name(self) -> str:
        return "Z-Score (μ=0, σ=1)"

class RobustNormalization(NormalizationStrategy):
    """Résistante aux outliers — utilise médiane et IQR"""
    def normalize(self, data: List[float]) -> List[float]:
        sorted_data = sorted(data)
        n = len(sorted_data)
        median = sorted_data[n // 2]
        q1 = sorted_data[n // 4]
        q3 = sorted_data[3 * n // 4]
        iqr = q3 - q1
        return [(x - median) / iqr if iqr > 0 else 0.0 for x in data]

    def name(self) -> str:
        return "Robust (median/IQR)"

# Contexte qui utilise la stratégie
class DataPreprocessor:
    def __init__(self, strategy: NormalizationStrategy):
        self._strategy = strategy

    @property
    def strategy(self) -> NormalizationStrategy:
        return self._strategy

    @strategy.setter
    def strategy(self, strategy: NormalizationStrategy):
        """Changer la stratégie à runtime"""
        print(f"Switching strategy to: {strategy.name()}")
        self._strategy = strategy

    def process(self, data: List[float]) -> List[float]:
        print(f"Using strategy: {self._strategy.name()}")
        return self._strategy.normalize(data)

# Usage
data = [1.0, 2.0, 3.0, 100.0, 5.0]  # outlier à 100
print(f"Raw data: {data}")

preprocessor = DataPreprocessor(MinMaxNormalization())
print(f"MinMax: {[round(x, 3) for x in preprocessor.process(data)]}")

preprocessor.strategy = ZScoreNormalization()
print(f"ZScore: {[round(x, 3) for x in preprocessor.process(data)]}")

preprocessor.strategy = RobustNormalization()
print(f"Robust: {[round(x, 3) for x in preprocessor.process(data)]}")


---
## 5. Design Pattern — Adapter <a name='adapter'></a>

> **But** : Permettre à des interfaces incompatibles de collaborer. Convertit l'interface d'une classe en une autre attendue par le client.

**Use case ML** : Adapter un modèle sklearn pour qu'il soit utilisable comme un modèle PyTorch, ou adapter une API externe.

In [ ]:
from abc import ABC, abstractmethod
import json
from typing import List, Dict, Any

# ============================================================
# ADAPTER PATTERN — Adapter différentes APIs de modèles
# ============================================================

# Interface cible (ce que notre système attend)
class MLModelInterface(ABC):
    @abstractmethod
    def train(self, features: List[List[float]], labels: List[int]) -> None: ...

    @abstractmethod
    def predict(self, features: List[List[float]]) -> List[int]: ...

    @abstractmethod
    def get_params(self) -> Dict[str, Any]: ...

# Classe existante (incompatible) — simule sklearn
class LegacySklearnModel:
    """API sklearn legacy — interface différente"""
    def fit(self, X, y, sample_weight=None):
        print(f"sklearn fit() called with {len(X)} samples")
        self._n_features = len(X[0]) if X else 0

    def predict_proba(self, X):
        return [[0.3, 0.7]] * len(X)

    def get_params(self, deep=True):
        return {"C": 1.0, "max_iter": 100}

# Adapter
class SklearnAdapter(MLModelInterface):
    """Adapte LegacySklearnModel vers MLModelInterface"""

    def __init__(self, sklearn_model: LegacySklearnModel):
        self._adaptee = sklearn_model

    def train(self, features: List[List[float]], labels: List[int]) -> None:
        # Traduit train() -> fit()
        self._adaptee.fit(features, labels)

    def predict(self, features: List[List[float]]) -> List[int]:
        # Traduit predict() -> predict_proba() + argmax
        proba = self._adaptee.predict_proba(features)
        return [max(range(len(p)), key=lambda i: p[i]) for p in proba]

    def get_params(self) -> Dict[str, Any]:
        return self._adaptee.get_params()

# Classe cliente — ne connaît que MLModelInterface
class ModelTrainer:
    def __init__(self, model: MLModelInterface):
        self.model = model

    def run_training(self, X, y):
        print(f"Trainer params: {self.model.get_params()}")
        self.model.train(X, y)
        predictions = self.model.predict(X[:3])
        print(f"Sample predictions: {predictions}")

# Usage — Le trainer ne sait pas qu'il utilise sklearn
legacy = LegacySklearnModel()
adapter = SklearnAdapter(legacy)
trainer = ModelTrainer(adapter)

X = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
y = [0, 1, 0]
trainer.run_training(X, y)


---
## 6. Design Pattern — Decorator <a name='decorator'></a>

> **But** : Ajouter dynamiquement des comportements à un objet sans modifier sa classe. Alternative flexible à l'héritage.

> ⚠️ À ne pas confondre avec les **décorateurs Python** (`@functools.wraps`) — bien que le principe soit similaire.

In [ ]:
import time
import functools
from abc import ABC, abstractmethod
from typing import List, Any
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================================
# DECORATOR PATTERN (Design Pattern GoF)
# ============================================================
class ModelComponent(ABC):
    @abstractmethod
    def predict(self, X: List[List[float]]) -> List[float]: ...

class BasePredictor(ModelComponent):
    def predict(self, X: List[List[float]]) -> List[float]:
        # Simulation d'une prédiction
        return [sum(row) / len(row) for row in X]

class ModelDecorator(ModelComponent):
    """Décorateur de base"""
    def __init__(self, component: ModelComponent):
        self._component = component

    def predict(self, X: List[List[float]]) -> List[float]:
        return self._component.predict(X)

class TimingDecorator(ModelDecorator):
    """Mesure le temps d'inférence"""
    def predict(self, X):
        start = time.perf_counter()
        result = super().predict(X)
        elapsed = time.perf_counter() - start
        print(f"[TIMING] Prediction took {elapsed*1000:.3f}ms for {len(X)} samples")
        return result

class LoggingDecorator(ModelDecorator):
    """Log les prédictions"""
    def predict(self, X):
        logger.info(f"[LOG] Predicting {len(X)} samples")
        result = super().predict(X)
        logger.info(f"[LOG] Results range: [{min(result):.2f}, {max(result):.2f}]")
        return result

class CachingDecorator(ModelDecorator):
    """Cache les résultats"""
    def __init__(self, component):
        super().__init__(component)
        self._cache = {}

    def predict(self, X):
        key = str(X)
        if key in self._cache:
            print("[CACHE] Cache hit!")
            return self._cache[key]
        result = super().predict(X)
        self._cache[key] = result
        print("[CACHE] Result cached")
        return result

# Composition de décorateurs (comme des couches)
X = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]

model = CachingDecorator(TimingDecorator(LoggingDecorator(BasePredictor())))
print("=== First call ===")
result = model.predict(X)
print(f"Result: {result}")
print("\n=== Second call (cached) ===")
result = model.predict(X)

# ============================================================
# DÉCORATEURS PYTHON (@functools.wraps) — très utilisés en ML
# ============================================================
def retry(max_attempts: int = 3, delay: float = 1.0):
    """Décorateur retry pour les appels API/modèles"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay)
        return wrapper
    return decorator

def validate_input(func):
    """Décorateur de validation pour les modèles"""
    @functools.wraps(func)
    def wrapper(self, X, *args, **kwargs):
        if not X or not all(isinstance(row, (list, tuple)) for row in X):
            raise ValueError("X must be a non-empty list of lists")
        return func(self, X, *args, **kwargs)
    return wrapper

class RobustPredictor:
    @validate_input
    @retry(max_attempts=2)
    def predict(self, X):
        return [1] * len(X)

rp = RobustPredictor()
print(rp.predict([[1, 2], [3, 4]]))


---
## 7. Design Pattern — Singleton <a name='singleton'></a>

> **But** : Garantir qu'une classe n'a qu'une seule instance et fournir un point d'accès global.

**Use case ML** : Configuration globale, connexion à une base de données, chargement de modèle unique.

In [ ]:
import threading
from typing import Optional, Dict, Any

# ============================================================
# SINGLETON — 3 implémentations Python
# ============================================================

# Version 1 : __new__
class ConfigSingleton:
    """Configuration globale de l'application ML"""
    _instance: Optional["ConfigSingleton"] = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._config = {
                "model_path": "/models/",
                "batch_size": 32,
                "device": "cpu",
                "log_level": "INFO"
            }
        return cls._instance

    def get(self, key: str, default=None):
        return self._config.get(key, default)

    def set(self, key: str, value: Any):
        self._config[key] = value

    def __repr__(self):
        return f"ConfigSingleton(id={id(self)}, config={self._config})"

c1 = ConfigSingleton()
c2 = ConfigSingleton()
c1.set("batch_size", 64)
print(f"c1 is c2: {c1 is c2}")  # True
print(f"c2.batch_size: {c2.get('batch_size')}")  # 64 — même instance

# Version 2 : Metaclass (plus élégant)
class SingletonMeta(type):
    _instances: Dict = {}
    _lock: threading.Lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        with cls._lock:  # Thread-safe
            if cls not in cls._instances:
                instance = super().__call__(*args, **kwargs)
                cls._instances[cls] = instance
        return cls._instances[cls]

class ModelRegistry(metaclass=SingletonMeta):
    """Registre unique de tous les modèles chargés"""
    def __init__(self):
        self._models: Dict[str, Any] = {}

    def register(self, name: str, model: Any) -> None:
        self._models[name] = model
        print(f"Model '{name}' registered")

    def get(self, name: str) -> Any:
        if name not in self._models:
            raise KeyError(f"Model '{name}' not found")
        return self._models[name]

    def list_models(self):
        return list(self._models.keys())

# Version 3 : Module-level (le plus pythonique)
# Dans un vrai projet, on ferait: config.py -> _config = {...} et des fonctions get/set

# Test thread-safety
registry1 = ModelRegistry()
registry2 = ModelRegistry()
registry1.register("bert-base", {"type": "transformer", "size": "110M"})
print(f"registry2.list_models(): {registry2.list_models()}")
print(f"registry1 is registry2: {registry1 is registry2}")


---
## 8. Design Pattern — Observer <a name='observer'></a>

> **But** : Définir une dépendance 1-N entre objets. Quand l'objet change, tous ses observateurs sont notifiés.

**Use case ML** : Monitoring d'entraînement, callbacks de training, pipeline d'événements.

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Dict, Any
from dataclasses import dataclass

# ============================================================
# OBSERVER PATTERN — Monitoring d'entraînement ML
# ============================================================
@dataclass
class TrainingEvent:
    epoch: int
    train_loss: float
    val_loss: float
    metrics: Dict[str, float]

class TrainingObserver(ABC):
    @abstractmethod
    def on_epoch_end(self, event: TrainingEvent) -> None: ...

class LossLogger(TrainingObserver):
    def on_epoch_end(self, event: TrainingEvent) -> None:
        print(f"[Logger] Epoch {event.epoch}: train_loss={event.train_loss:.4f}, val_loss={event.val_loss:.4f}")

class EarlyStopping(TrainingObserver):
    def __init__(self, patience: int = 3):
        self.patience = patience
        self.best_val_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def on_epoch_end(self, event: TrainingEvent) -> None:
        if event.val_loss < self.best_val_loss:
            self.best_val_loss = event.val_loss
            self.counter = 0
        else:
            self.counter += 1
            print(f"[EarlyStopping] No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.should_stop = True
                print("[EarlyStopping] ⛔ Stopping training!")

class ModelCheckpoint(TrainingObserver):
    def __init__(self, save_path: str, monitor: str = "val_loss"):
        self.save_path = save_path
        self.monitor = monitor
        self.best_value = float("inf")

    def on_epoch_end(self, event: TrainingEvent) -> None:
        current = event.val_loss if self.monitor == "val_loss" else event.metrics.get(self.monitor, float("inf"))
        if current < self.best_value:
            self.best_value = current
            print(f"[Checkpoint] ✅ Saving model to {self.save_path} (val_loss={current:.4f})")

class Trainer:
    """Subject — notifie les observers"""
    def __init__(self):
        self._observers: List[TrainingObserver] = []

    def attach(self, observer: TrainingObserver) -> None:
        self._observers.append(observer)

    def detach(self, observer: TrainingObserver) -> None:
        self._observers.remove(observer)

    def _notify(self, event: TrainingEvent) -> None:
        for obs in self._observers:
            obs.on_epoch_end(event)

    def fit(self, epochs: int = 5):
        import math
        for epoch in range(1, epochs + 1):
            # Simulation d'entraînement
            train_loss = 1.0 / epoch + 0.01
            val_loss = 1.0 / epoch + 0.05 + (0.1 if epoch > 3 else 0)
            event = TrainingEvent(epoch, train_loss, val_loss, {"accuracy": 0.9 - 0.01*epoch})
            self._notify(event)

early_stop = EarlyStopping(patience=2)
trainer = Trainer()
trainer.attach(LossLogger())
trainer.attach(early_stop)
trainer.attach(ModelCheckpoint("/checkpoints/best_model.pt"))
trainer.fit(epochs=6)


---
## 9. Principes SOLID <a name='solid'></a>

In [ ]:
# ============================================================
# S — Single Responsibility Principle
# Une classe = une responsabilité
# ============================================================

# ❌ MAL : Une classe fait tout
class BadModelManager:
    def train(self, X, y): pass
    def predict(self, X): pass
    def save_to_db(self): pass      # Responsabilité DB
    def send_email_report(self): pass  # Responsabilité notification
    def log_to_file(self): pass     # Responsabilité logging

# ✅ BIEN : Séparation des responsabilités
class Model:
    def train(self, X, y): print("Training...")
    def predict(self, X): return [0] * len(X)

class ModelPersistence:
    def save(self, model: Model, path: str): print(f"Saving to {path}")
    def load(self, path: str) -> Model: return Model()

class ModelReporter:
    def generate_report(self, model: Model) -> str: return "Report..."

# ============================================================
# O — Open/Closed Principle
# Ouvert à l'extension, fermé à la modification
# ============================================================
from abc import ABC, abstractmethod

class EvaluationMetric(ABC):
    @abstractmethod
    def compute(self, y_true, y_pred) -> float: ...

class AccuracyMetric(EvaluationMetric):
    def compute(self, y_true, y_pred) -> float:
        return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

class F1Metric(EvaluationMetric):
    def compute(self, y_true, y_pred) -> float:
        tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
        fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
        fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        return 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

class Evaluator:
    """Pas besoin de modifier cette classe pour ajouter des métriques"""
    def __init__(self, metrics: list):
        self.metrics = metrics

    def evaluate(self, y_true, y_pred):
        return {type(m).__name__: m.compute(y_true, y_pred) for m in self.metrics}

# Ajouter une nouvelle métrique = créer une nouvelle classe, pas modifier Evaluator
y_true = [1, 0, 1, 1, 0]
y_pred = [1, 0, 0, 1, 0]
evaluator = Evaluator([AccuracyMetric(), F1Metric()])
print(evaluator.evaluate(y_true, y_pred))

# ============================================================
# L — Liskov Substitution Principle
# Une sous-classe doit pouvoir remplacer sa classe parent
# ============================================================
# D — Dependency Inversion Principle
# Dépendre d'abstractions, pas d'implémentations concrètes
# ============================================================
class DataRepository(ABC):
    @abstractmethod
    def get_training_data(self): ...

class PostgresRepository(DataRepository):
    def get_training_data(self): return {"source": "postgres"}

class S3Repository(DataRepository):
    def get_training_data(self): return {"source": "s3"}

class TrainingPipeline:
    """Dépend de l'abstraction DataRepository, pas d'une DB spécifique"""
    def __init__(self, repository: DataRepository):
        self.repository = repository  # Injection de dépendance

    def run(self):
        data = self.repository.get_training_data()
        print(f"Training with data from: {data['source']}")

# Facile à changer ou mocker pour les tests
pipeline = TrainingPipeline(S3Repository())
pipeline.run()
pipeline = TrainingPipeline(PostgresRepository())
pipeline.run()


---
## 10. Dataclasses & Protocols <a name='dataclasses'></a>

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Protocol
import json

# ============================================================
# DATACLASSES — Très utilisées pour les configs ML
# ============================================================
@dataclass
class TrainingConfig:
    model_name: str
    learning_rate: float = 1e-4
    batch_size: int = 32
    epochs: int = 10
    optimizer: str = "adam"
    device: str = "cpu"
    dropout: float = 0.1
    layers: List[int] = field(default_factory=lambda: [256, 128, 64])
    tags: List[str] = field(default_factory=list)

    def __post_init__(self):
        """Validation après initialisation"""
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive")
        if self.batch_size <= 0:
            raise ValueError("batch_size must be positive")

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)

    @classmethod
    def from_json(cls, json_str: str) -> "TrainingConfig":
        return cls(**json.loads(json_str))

config = TrainingConfig(
    model_name="bert-base-uncased",
    learning_rate=2e-5,
    epochs=5,
    tags=["nlp", "classification"]
)
print(config)
print(f"\nJSON:\n{config.to_json()}")

# Dataclass frozen (immutable) — utile pour les hyperparams
from dataclasses import dataclass

@dataclass(frozen=True)
class HyperParams:
    lr: float
    momentum: float
    weight_decay: float

hp = HyperParams(lr=0.01, momentum=0.9, weight_decay=1e-4)
print(f"\nFrozen HyperParams: {hp}")
# hp.lr = 0.1  # FrozenInstanceError
print(f"Hashable (utilisable comme clé dict): {hash(hp)}")


---
## 11. Context Managers & Generators <a name='context'></a>

In [ ]:
from contextlib import contextmanager
import time
from typing import Generator, Iterator

# ============================================================
# CONTEXT MANAGER — Gestion des ressources
# ============================================================
class ModelInference:
    """Context manager pour l'inférence (charge/décharge le modèle)"""
    def __init__(self, model_path: str):
        self.model_path = model_path
        self.model = None

    def __enter__(self):
        print(f"Loading model from {self.model_path}...")
        self.model = {"loaded": True, "path": self.model_path}  # Simule le chargement
        return self.model

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Unloading model from memory...")
        self.model = None
        if exc_type:
            print(f"Exception occurred: {exc_type.__name__}: {exc_val}")
        return False  # Ne supprime pas l'exception

with ModelInference("/models/gpt2") as model:
    print(f"Predicting with model: {model}")

# Version avec @contextmanager
@contextmanager
def timer(label: str = ""):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Elapsed: {elapsed*1000:.3f}ms")

with timer("Matrix multiplication"):
    result = sum(i*j for i in range(1000) for j in range(100))

# ============================================================
# GENERATORS — Traitement de données par batch (memory-efficient)
# ============================================================
def batch_generator(data: list, batch_size: int) -> Generator:
    """Génère des batches sans charger tout en mémoire"""
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def infinite_stream(start: int = 0) -> Iterator[int]:
    """Générateur infini — utile pour streaming"""
    n = start
    while True:
        yield n
        n += 1

# Traitement de 1M d'exemples par batch de 1000
data = list(range(10_000))
total_batches = 0
for batch in batch_generator(data, batch_size=1000):
    total_batches += 1
    # Traitement du batch...

print(f"Processed {len(data)} items in {total_batches} batches")

# Generator expressions — lazy evaluation
import sys
list_comp = [x**2 for x in range(10_000)]
gen_exp = (x**2 for x in range(10_000))
print(f"List memory: {sys.getsizeof(list_comp):,} bytes")
print(f"Generator memory: {sys.getsizeof(gen_exp):,} bytes")


---
## 13. Questions d'entretien fréquentes <a name='questions'></a>

### 🎯 Questions POO classiques BNP/Finance

**Q: Quelle est la différence entre `__str__` et `__repr__` ?**
> `__repr__` : représentation non ambiguë pour les développeurs (eval-able si possible). `__str__` : représentation lisible pour les utilisateurs. Si seul `__repr__` est défini, il sert aussi de `__str__`.

**Q: Qu'est-ce que le MRO (Method Resolution Order) ?**
> L'ordre dans lequel Python cherche les méthodes dans la hiérarchie de classes. Utilisé avec l'héritage multiple. Python utilise l'algorithme C3 linearization. `MyClass.__mro__` pour voir l'ordre.

**Q: `@classmethod` vs `@staticmethod` ?**
> `@classmethod` : reçoit `cls` comme premier argument, peut accéder/modifier l'état de la classe. `@staticmethod` : ne reçoit ni `self` ni `cls`, c'est juste une fonction dans l'espace de noms de la classe.

**Q: Quand utiliser Composition vs Héritage ?**
> Règle : "favor composition over inheritance". Héritage = relation "est-un" (is-a). Composition = relation "a-un" (has-a). Composition est plus flexible et évite le couplage fort.

**Q: Quand utiliser Factory vs Strategy ?**
> Factory : problème de *création* d'objets (quel type instancier ?). Strategy : problème de *comportement* (quel algorithme utiliser ?).

**Q: Comment implémenter un Singleton thread-safe en Python ?**
> Utiliser un `threading.Lock` dans la metaclass ou `__new__`. Attention, le GIL ne suffit pas pour garantir la thread-safety du singleton.

**Q: Qu'est-ce que `__slots__` ?**
> Remplace le dictionnaire `__dict__` par un tuple fixe d'attributs. Réduit la mémoire (~40%) et accélère l'accès aux attributs. À utiliser quand on crée des millions d'instances.


In [ ]:
# MRO Example
class A:
    def method(self): return "A"

class B(A):
    def method(self): return "B -> " + super().method()

class C(A):
    def method(self): return "C -> " + super().method()

class D(B, C):
    def method(self): return "D -> " + super().method()

print(D.__mro__)
print(D().method())  # D -> B -> C -> A

# __slots__
class WithDict:
    def __init__(self, x, y): self.x = x; self.y = y

class WithSlots:
    __slots__ = ['x', 'y']
    def __init__(self, x, y): self.x = x; self.y = y

import sys
print(f"With __dict__: {sys.getsizeof(WithDict(1, 2))} bytes")
print(f"With __slots__: {sys.getsizeof(WithSlots(1, 2))} bytes")

# classmethod vs staticmethod
class ModelUtils:
    _count = 0

    def __init__(self):
        ModelUtils._count += 1

    @classmethod
    def get_count(cls) -> int:
        """Accède à l'état de la classe"""
        return cls._count

    @classmethod
    def create_with_defaults(cls) -> "ModelUtils":
        """Factory method via classmethod"""
        return cls()

    @staticmethod
    def validate_lr(lr: float) -> bool:
        """Pas besoin de l'état de la classe"""
        return 0 < lr < 1

ModelUtils()
ModelUtils()
print(f"Instances created: {ModelUtils.get_count()}")
print(f"LR valid: {ModelUtils.validate_lr(0.001)}")
